In [ ]:
import requests
import pandas as pd
import time
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import matplotlib.pyplot as plt

from google.colab import userdata

# ── CONFIG ───────────────────────────────────────────────────────────────────
API_KEY = userdata.get('RIOT_API_KEY')
REGION  = "na1"
REGION_ROUTING = "americas"
GAMES_TO_COLLECT = 2000         # aim high, collect whatever we can before key dies
SAVE_EVERY = 50                 # save progress to CSV every N games
SAVE_PATH = "lol_games.csv"     # will appear in your Colab files panel on the left
HEADERS = {"X-Riot-Token": API_KEY}

def get(url):
    time.sleep(1.3)
    r = requests.get(url, headers=HEADERS)
    if r.status_code == 403:
        raise Exception("API key expired or invalid — regenerate it at developer.riot.games")
    if r.status_code == 429:
        print("Rate limited, sleeping 60s...")
        time.sleep(60)
        return get(url)
    return r.json()


# ── 1. LOAD EXISTING PROGRESS IF RESUMING ───────────────────────────────────
# If lol_games.csv already exists (from a previous run), load it so we don't
# re-collect games we already have.
if os.path.exists(SAVE_PATH):
    df_existing = pd.read_csv(SAVE_PATH)
    rows = df_existing.to_dict("records")
    print(f"Resuming — loaded {len(rows)} existing games from {SAVE_PATH}")
else:
    rows = []
    print("Starting fresh")


# ── 2. GET PLAYERS FROM EVERY RANK TIER ─────────────────────────────────────
print("Fetching players...")
puuids = []

tiers = ["IRON","BRONZE","SILVER","GOLD","PLATINUM","EMERALD","DIAMOND"]
for tier in tiers:
    for division in ["I","II","III","IV"]:
        url = f"https://{REGION}.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{division}?page=1"
        entries = get(url)
        if isinstance(entries, list):
            for entry in entries[:10]:
                summoner_url = f"https://{REGION}.api.riotgames.com/lol/summoner/v4/summoners/{entry['summonerId']}"
                summoner = get(summoner_url)
                if "puuid" in summoner:
                    puuids.append(summoner["puuid"])

print(f"Got {len(puuids)} players")


# ── 3. GET MATCH IDs ─────────────────────────────────────────────────────────
print("Fetching match IDs...")
match_ids = set()

for puuid in puuids:
    if len(match_ids) >= GAMES_TO_COLLECT:
        break
    url = f"https://{REGION_ROUTING}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?queue=420&count=10"
    ids = get(url)
    if isinstance(ids, list):
        match_ids.update(ids)

# remove match IDs we already collected in a previous run
if rows:
    existing_ids = set()   # we don't store match_id in rows, so just collect what's new
match_ids = list(match_ids)[:GAMES_TO_COLLECT]
print(f"Got {len(match_ids)} match IDs to collect")


# ── 4. FETCH EACH MATCH AND SAVE PROGRESS AS WE GO ──────────────────────────
print("Fetching match details — this will take a while, progress saves every 50 games...")

for i, match_id in enumerate(match_ids):

    # skip if we already have enough
    if len(rows) >= GAMES_TO_COLLECT:
        break

    if i % 50 == 0:
        print(f"  Match {i}/{len(match_ids)} | Games collected so far: {len(rows)}")

    url = f"https://{REGION_ROUTING}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    match = get(url)

    if "info" not in match:
        continue

    info = match["info"]
    participants = info["participants"]

    if len(participants) != 10:
        continue

    row = {}
    team1 = [p for p in participants if p["teamId"] == 100]
    team2 = [p for p in participants if p["teamId"] == 200]

    for j, p in enumerate(team1, 1):
        row[f"t1_champ{j}id"] = p["championId"]
    for j, p in enumerate(team2, 1):
        row[f"t2_champ{j}id"] = p["championId"]

    row["winner"] = 1 if team1[0]["win"] else 2
    rows.append(row)

    # ── SAVE CHECKPOINT EVERY N GAMES ───────────────────────────────────────
    if len(rows) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(SAVE_PATH, index=False)
        print(f"  💾 Saved {len(rows)} games to {SAVE_PATH}")

# final save when done
pd.DataFrame(rows).to_csv(SAVE_PATH, index=False)
print(f"Done! Total games collected: {len(rows)} — saved to {SAVE_PATH}")
df_games = pd.DataFrame(rows)


# ── 5. BUILD CHAMPION PRESENCE FEATURES ─────────────────────────────────────
print("Building features...")
all_champ_ids = sorted(
    set(df_games[[f"t1_champ{i}id" for i in range(1,6)]].values.flatten()) |
    set(df_games[[f"t2_champ{i}id" for i in range(1,6)]].values.flatten())
)

game_features = []
for _, row in df_games.iterrows():
    game_dict = {champ_id: 0 for champ_id in all_champ_ids}
    for i in range(1, 6):
        game_dict[row[f"t1_champ{i}id"]] = 1
        game_dict[row[f"t2_champ{i}id"]] = -1
    game_dict["winner"] = row["winner"]
    game_features.append(game_dict)

df_champion_presence = pd.DataFrame(game_features)
print("Champion presence dataframe:", df_champion_presence.shape)


# ── 6. TRAIN ─────────────────────────────────────────────────────────────────
X = df_champion_presence.drop("winner", axis=1)
y = df_champion_presence["winner"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

params = {
    "n_estimators":     [20, 50, 100],
    "max_depth":        [5, 10, 15],
    "min_samples_leaf": [1, 5, 10]
}

grid_clf = GridSearchCV(
    RandomForestClassifier(random_state=0),
    param_grid=params,
    return_train_score=True,
    verbose=1
)
grid_clf.fit(X_train, y_train)

print("Best params:", grid_clf.best_params_)
print(f"Best validation score: {grid_clf.best_score_:.3f}")
print(f"Test score:            {grid_clf.score(X_test, y_test):.3f}")

SyntaxError: invalid decimal literal (4040880753.py, line 17)